In [7]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../../data/education.db")

In [9]:
conn.executescript("""
DROP TABLE IF EXISTS tertiary_female_raw;
DROP TABLE IF EXISTS tertiary_male_raw;
DROP TABLE IF EXISTS tertiary_female_clean;
DROP TABLE IF EXISTS tertiary_male_clean;
DROP TABLE IF EXISTS tertiary_gender_merged;
DROP TABLE IF EXISTS tertiary_male_long_sql;
DROP TABLE IF EXISTS tertiary_female_long_sql;
""")

In [11]:
df_tertiary_female = pd.read_csv(
    "../../data/raw/enrollment/TERTIARY Female School Enrollment % DATA (3).csv",
    skiprows=4,
    engine="python"
)

df_tertiary_female.to_sql(
    "tertiary_female_raw",
    conn,
    if_exists="replace",
    index=False
)

pd.read_sql("SELECT COUNT(*) FROM tertiary_female_raw;", conn)

,COUNT(*)
0,266


In [13]:
df_tertiary_male = pd.read_csv(
    "../../data/raw/enrollment/TERTIARY Male School Enrollment % DATA.csv.csv",
    skiprows=4,
    engine="python"
)

df_tertiary_male.head()


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"School enrollment, tertiary, male (% gross)",SE.TER.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.638906,NaN,NaN
1,Africa Eastern and Southern,AFE,"School enrollment, tertiary, male (% gross)",SE.TER.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,9.49413,9.48387,9.37550,9.36714,9.30325,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,"School enrollment, tertiary, male (% gross)",SE.TER.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,14.77660,NaN,15.58478,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,"School enrollment, tertiary, male (% gross)",SE.TER.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,11.35700,11.46254,11.48615,11.65789,11.84946,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,"School enrollment, tertiary, male (% gross)",SE.TER.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,10.1656,10.011618,NaN,NaN


In [14]:
df_tertiary_male_clean = df_tertiary_male[
    ["Country Name", "Country Code", "Indicator Name",
     "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
     "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
     "2020","2021","2022","2023"]
]

df_tertiary_male_clean = df_tertiary_male_clean.dropna(how="all", subset=[
    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"
])

df_tertiary_male_clean.to_sql(
    "tertiary_male_raw",
    conn,
    if_exists="replace",
    index=False
)

238

In [15]:
conn.executescript("""
DROP TABLE IF EXISTS tertiary_female_clean;
DROP TABLE IF EXISTS tertiary_male_clean;
""")

In [16]:
conn.executescript("""
CREATE TABLE tertiary_female_clean AS
SELECT
    "Country Name" AS country_name,
    "Country Code" AS country_code,
    "Indicator Name" AS indicator_name,

    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"

FROM tertiary_female_raw

WHERE "Country Code" IN (
'AFE','AFW','ARB','AUS','EAS','EUU','LCN','NAC','SAS',
'LIC','LMC','UMC','HIC'
);
""")

pd.read_sql("SELECT COUNT(*) FROM tertiary_female_clean;", conn)

,COUNT(*)
0,13


In [17]:
conn.executescript("""
CREATE TABLE tertiary_male_clean AS
SELECT
    "Country Name" AS country_name,
    TRIM("Country Code") AS country_code,
    "Indicator Name" AS indicator_name,

    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"

FROM tertiary_male_raw
WHERE TRIM("Country Code") IN (
'AFE','AFW','ARB','AUS','EAS','EUU','LCN','NAC','SAS',
'LIC','LMC','UMC','HIC'
);
""")

pd.read_sql("SELECT COUNT(*) FROM tertiary_male_clean;", conn)

,COUNT(*)
0,13


In [18]:
pd.read_sql("""
SELECT
  'male' AS table_name,
  COUNT(*) AS rows,
  MIN("2000") AS min_2000,
  MAX("2000") AS max_2000
FROM tertiary_male_clean

UNION ALL

SELECT
  'female',
  COUNT(*),
  MIN("2000"),
  MAX("2000")
FROM tertiary_female_clean;
""", conn)

,table_name,rows,min_2000,max_2000
0,male,13,4.49486,57.813068
1,female,13,3.08268,76.430763


In [20]:
conn.executescript("""
DROP TABLE IF EXISTS tertiary_male_long_sql;
DROP TABLE IF EXISTS tertiary_female_long_sql;

CREATE TABLE tertiary_male_long_sql AS
SELECT country_name, country_code, '2000' AS year, "2000" AS male_enrollment FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2001', "2001" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2002', "2002" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2003', "2003" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2004', "2004" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2005', "2005" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2006', "2006" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2007', "2007" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2008', "2008" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2009', "2009" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2010', "2010" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2011', "2011" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2012', "2012" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2013', "2013" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2014', "2014" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2015', "2015" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2016', "2016" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2017', "2017" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2018', "2018" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2019', "2019" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2020', "2020" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2021', "2021" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2022', "2022" FROM tertiary_male_clean
UNION ALL SELECT country_name, country_code, '2023', "2023" FROM tertiary_male_clean;

CREATE TABLE tertiary_female_long_sql AS
SELECT country_name, country_code, '2000' AS year, "2000" AS female_enrollment FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2001', "2001" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2002', "2002" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2003', "2003" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2004', "2004" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2005', "2005" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2006', "2006" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2007', "2007" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2008', "2008" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2009', "2009" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2010', "2010" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2011', "2011" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2012', "2012" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2013', "2013" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2014', "2014" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2015', "2015" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2016', "2016" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2017', "2017" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2018', "2018" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2019', "2019" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2020', "2020" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2021', "2021" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2022', "2022" FROM tertiary_female_clean
UNION ALL SELECT country_name, country_code, '2023', "2023" FROM tertiary_female_clean;
""")

In [21]:
conn.executescript("""
DROP TABLE IF EXISTS tertiary_gender_merged;

CREATE TABLE tertiary_gender_merged AS
SELECT
    m.country_name,
    m.country_code,
    m.year,
    m.male_enrollment,
    f.female_enrollment
FROM tertiary_male_long_sql m
JOIN tertiary_female_long_sql f
    ON m.country_code = f.country_code
    AND m.year = f.year;
""")

In [22]:
pd.read_sql("SELECT COUNT(*) FROM tertiary_gender_merged;", conn)

,COUNT(*)
0,312


In [23]:
pd.read_sql("SELECT * FROM tertiary_gender_merged LIMIT 10;", conn)

,country_name,country_code,year,male_enrollment,female_enrollment
0,Africa Eastern and Southern,AFE,2000,4.494860,3.504690
1,Africa Western and Central,AFW,2000,6.158440,3.729520
2,Arab World,ARB,2000,17.508869,16.152531
3,Australia,AUS,2000,NaN,NaN
4,East Asia & Pacific,EAS,2000,17.852610,15.196400
5,European Union,EUU,2000,47.276020,54.674171
6,High income,HIC,2000,52.466240,61.032410
7,Latin America & Caribbean,LCN,2000,21.870270,25.767040
8,Low income,LIC,2000,5.882230,3.082680
9,Lower middle income,LMC,2000,11.149310,8.307320


In [25]:
df = pd.read_sql("SELECT * FROM tertiary_gender_merged;", conn)

df.to_csv("../../data/cleaned/tertiary_gender_merged.csv", index=False)

pd.read_sql("SELECT * FROM tertiary_male_clean;", conn).to_csv(
    "../../data/cleaned/tertiary_male_clean.csv", index=False
)

pd.read_sql("SELECT * FROM tertiary_female_clean;", conn).to_csv(
    "../../data/cleaned/tertiary_female_clean.csv", index=False
)

In [28]:
import os
os.listdir("../../data/cleaned")

['tertiary_female_clean.csv',
 'all_enrollment_combined.csv',
 '.gitkeep',
 'primary_female_clean.csv',
 'tertiary_gender_merged.csv',
 'secondary_female_clean.csv',
 'primary_male_clean.csv',
 'total_trained_teachers_secondary_cleaned.csv',
 'secondary_gender_merged.csv',
 'primary_gender_merged.csv',
 'secondary_male_clean.csv',
 'tertiary_male_clean.csv']